In [0]:
%pip install "ray[default]" "ray[train]" transformers datasets pytorch-lightning
dbutils.library.restartPython()

No Lightning test

In [0]:
from ray.train.torch import TorchTrainer
from ray.air.config import ScalingConfig

def train_loop_per_worker(config):
    import os, socket, torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader
    from datasets import load_dataset
    from transformers import GPT2LMHeadModel, GPT2Tokenizer

    torch.set_float32_matmul_precision("high")
    device = torch.device("cuda", 0)

    # tiny dataset (fully local inside function)
    ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:1%]")
    tok = GPT2Tokenizer.from_pretrained("gpt2")
    tok.pad_token = tok.eos_token

    def enc(ex):
        out = tok(ex["text"], truncation=True, padding="max_length", max_length=64, return_tensors=None)
        return {"input_ids": out["input_ids"]}
    ds = ds.map(enc, batched=True).with_format("torch")

    dl = DataLoader(ds, batch_size=2, shuffle=True, num_workers=0)

    # HF model, plain torch
    model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
    model.train()
    opt = optim.AdamW(model.parameters(), lr=5e-5)

    print(f"🚀 host={socket.gethostname()} gpu={torch.cuda.get_device_name(0)}")
    it = iter(dl)
    for step in range(20):  # tiny smoke test
        batch = next(it)
        input_ids = batch["input_ids"].to(device)
        out = model(input_ids=input_ids, labels=input_ids)
        loss = out.loss
        loss.backward()
        opt.step()
        opt.zero_grad()
        if step % 5 == 0:
            print(f"[{socket.gethostname()}] step {step} loss {loss.item():.4f}")

scaling = ScalingConfig(num_workers=4, use_gpu=True)
trainer = TorchTrainer(train_loop_per_worker=train_loop_per_worker, scaling_config=scaling)
result = trainer.fit()
print("✅ Finished:", result)

In [0]:
import socket, os

driver_ip = spark.conf.get("spark.driver.host")
print("Spark driver host:", driver_ip)


# Start Ray head on the driver node
os.system(f"ray stop --force")  # cleanup if any Ray is already running
os.system(f"ray start --head --node-ip-address={driver_ip} --port=6379")

# Function for workers to join Ray
def start_ray_worker(_):
    import os
    os.system(f"ray stop --force")
    os.system(f"ray start --address={driver_ip}:6379")
    return ["worker joined"]

# Launch Ray on all workers using Spark barrier mode
spark.sparkContext.parallelize(range(3), 3).barrier().mapPartitions(start_ray_worker).collect()

print("Ray head + workers started.")

In [0]:
!nvidia-smi -L
!echo "CUDA_VISIBLE_DEVICES=$CUDA_VISIBLE_DEVICES"
!hostname

In [0]:
import torch
print("GPUs visible to PyTorch:", torch.cuda.device_count())
print("Device names:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [0]:
spark.conf.get("spark.databricks.clusterUsageTags.clusterTargetWorkers")

In [0]:
!nvidia-smi -L
import torch, ray
print("Visible GPUs:", torch.cuda.device_count())
print("Cluster resources:", ray.cluster_resources())

In [0]:
%pip install ray[default]==2.34.0
dbutils.library.restartPython()

In [0]:
from databricks.ray import RayCluster

ray_cluster = RayCluster(
    num_worker_nodes=3,
    num_gpus_per_node=4,
    head_node_options={"num_gpus": 4},
)
ray_context = ray_cluster.start()
print(ray_context.cluster_resources())